# Loss Given Default (LGD), Exposure at Default (EAD), and Expected Loss

## Objective

This notebook extends the Probability of Default (PD) model by estimating the remaining components of credit risk:

- Exposure at Default (EAD)
- Loss Given Default (LGD)
- Expected Loss (EL)

These metrics are widely used in banking for:

- credit risk assessment
- loan pricing
- portfolio monitoring
- capital allocation
- Expected Credit Loss (ECL) estimation

The notebook also evaluates expected losses across borrower risk bands, loan purposes, and credit grades.

In [3]:
import pandas as pd
import numpy as np

In [4]:
pd.set_option('display.max_columns',150)
pd.set_option('display.max_rows',100)

## Import Risk Segmentation Output

The risk-segmented borrower dataset generated in the previous notebook is imported.

This dataset already contains:

- PD score
- Risk band
- Risk score
- Preliminary loan decision

These values will now be combined with repayment information to estimate expected losses.

In [5]:
risk_df=pd.read_pickle('../data/risk_segmented_output.pkl')
risk_df.columns.to_list()

['loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'purpose',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'inq_last_6mths',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'total_rec_late_fee',
 'last_pymnt_d',
 'last_credit_pull_d',
 'collections_12_mths_ex_med',
 'application_type',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'total_rev_hi_lim',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'mo_sin_old_il_acct',
 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op',
 'mo_sin_rcnt_tl',
 'mort_acc',
 'mths_since_recent_bc',
 'mths_since_recent_inq',
 'num_accts_ever_120_pd',
 'num_actv_bc_tl',
 'num_actv_rev_tl',
 'num_bc_sats',
 'num_bc_tl',
 'num_il_tl',
 'num_op_rev_tl',
 'num_rev_accts',

In [6]:
risk_df[['actual_default_flag','pd_score','risk_band','risk_score']].tail(10)

,actual_default_flag,pd_score,risk_band,risk_score
692153,0,0.468976,Medium,531.0
1816200,0,0.233568,Low,766.0
1270714,0,0.050490,Very Low,950.0
1709748,0,0.533128,High,467.0
596968,0,0.739552,Very High,260.0
1135476,1,0.222382,Very Low,778.0
1601713,1,0.950497,Very High,50.0
344609,0,0.146255,Very Low,854.0
874428,1,0.554675,High,445.0
1910437,1,0.368561,Medium,631.0


In [7]:
risk_df.shape

(40000, 124)

## Import Full Lending Club Dataset

The complete feature-engineered Lending Club dataset is loaded.

This dataset contains repayment and recovery variables that are required to estimate:

- Exposure at Default (EAD)
- Realized Loss
- Loss Given Default (LGD)

These variables were intentionally excluded from the PD model to prevent target leakage but are now required for loss estimation.

In [8]:
full_df=pd.read_pickle('../data/lending_club_feature_engineered.pkl')
full_df.shape

(1340973, 128)

In [9]:
full_df.columns.to_list()

['loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'purpose',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'inq_last_6mths',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'last_credit_pull_d',
 'collections_12_mths_ex_med',
 'application_type',
 'acc_now_delinq',
 'tot_coll_amt',
 'tot_cur_bal',
 'total_rev_hi_lim',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'chargeoff_within_12_mths',
 'delinq_amnt',
 'mo_sin_old_il_acct',
 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op',
 'mo_sin_rcnt_tl',
 'mort_acc',
 'mths_since_recent_bc',
 

## Select Recovery Variables

Recovery-related variables are selected from the full dataset.

These variables include:

- funded amount
- principal recovered
- recoveries
- collection recovery fees
- loan grade
- interest rate
- loan purpose

These features are necessary for estimating borrower losses after default.

## Merge Recovery Information

Recovery-related variables are merged into the risk-segmented dataset.

This combines:

- borrower risk predictions
- repayment information
- recovery information

into one dataset for Expected Loss estimation.

In [14]:
recovery_cols = [
    "funded_amnt",
    "funded_amnt_clean",
    "total_rec_prncp",
    "recoveries",
    "collection_recovery_fee",
    "int_rate",
    "grade",
    "purpose"
]

available_recovery_cols = [cols for cols in recovery_cols if cols in full_df.columns]
print(available_recovery_cols)

risk_df = risk_df.join(full_df[available_recovery_cols], rsuffix="_orig")
print(risk_df.shape)

['funded_amnt', 'funded_amnt_clean', 'total_rec_prncp', 'recoveries', 'collection_recovery_fee', 'int_rate', 'grade', 'purpose']
(40000, 140)


In [15]:
risk_df[available_recovery_cols].head()

,funded_amnt,funded_amnt_clean,total_rec_prncp,recoveries,collection_recovery_fee,int_rate,grade,purpose
824713,12000,12000,5138.16,0.0,0.0,19.89,E,debt_consolidation
591446,20100,20100,20100.00,0.0,0.0,15.59,C,debt_consolidation
1789024,8000,8000,8000.00,0.0,0.0,11.14,B,credit_card
1084390,23325,23325,23325.00,0.0,0.0,12.69,C,credit_card
1807296,21250,21250,21250.00,0.0,0.0,22.95,F,debt_consolidation


## Estimate Exposure at Default (EAD)

Exposure at Default (EAD) represents the amount outstanding when a borrower defaults.

For this project, EAD is approximated using the funded loan amount.

Missing values are replaced where possible, and zero values are treated as missing to avoid incorrect calculations.

In [16]:
risk_df['ead'] = risk_df['funded_amnt_clean']

risk_df['ead'] = risk_df['ead'].fillna(risk_df['funded_amnt'])
risk_df['ead'] = risk_df['ead'].replace(0, np.nan)

risk_df['ead'].describe()

count    40000.000000
mean     14513.557500
std       8711.779018
min        500.000000
25%       8000.000000
50%      12000.000000
75%      20000.000000
max      40000.000000
Name: ead, dtype: float64

## Calculate Realized Loss

Realized Loss is calculated as:

Realized Loss = EAD − Principal Recovered − Recoveries

Negative losses are clipped to zero because recovered amounts cannot exceed the total exposure for loss calculation purposes.

In [18]:
risk_df['realized_loss'] = risk_df['ead'] - risk_df['total_rec_prncp'].fillna(0) - risk_df['recoveries'].fillna(0)

risk_df['realized_loss'] = risk_df['realized_loss'].clip(lower=0)

risk_df[['ead', 'total_rec_prncp', 'recoveries', 'realized_loss']].head()

,ead,total_rec_prncp,recoveries,realized_loss
824713,12000,5138.16,0.0,6861.84
591446,20100,20100.00,0.0,0.00
1789024,8000,8000.00,0.0,0.00
1084390,23325,23325.00,0.0,0.00
1807296,21250,21250.00,0.0,0.00


## Calculate Realized Loss Given Default (LGD)

Loss Given Default (LGD) measures the proportion of exposure that was ultimately lost.

Formula:

LGD = Realized Loss / EAD

The calculated values are constrained between 0 and 1 for valid probability interpretation.

In [59]:
risk_df['realized_lgd'] = risk_df['realized_loss']/risk_df['ead'] 
risk_df["realized_lgd"] = risk_df['realized_lgd'].replace([np.inf,-np.inf], np.nan)\
    .fillna(0)\
    .clip(lower=0, upper=1)

risk_df['realized_lgd'].describe()

count    40000.000000
mean         0.138327
std          0.279356
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: realized_lgd, dtype: float64

## Estimate LGD from Defaulted Borrowers

Only borrowers who actually defaulted are used for LGD estimation.

This provides:

- Global average LGD
- Grade-specific LGD estimates

Grade-level averages allow the model to capture differences in recovery behavior across credit grades.

In [61]:
defaulted_loans = risk_df[risk_df['actual_default_flag'] == 1].copy()

global_lgd = defaulted_loans['realized_lgd'].mean()
print(f'Global LGD Assumption : {global_lgd}')

Global LGD Assumption : 0.6245033211675892


## Grade-Level LGD Summary

Average realized LGD is calculated for each Lending Club credit grade.

The summary includes:

- number of defaulted loans
- average realized LGD
- average realized loss
- average exposure at default

Higher credit grades are expected to have lower realized losses than lower grades.

In [63]:
lgd_by_grade = defaulted_loans.groupby('grade')\
    .agg(
        defaulted_loan_count = ('actual_default_flag','count'),
        avg_realized_lgd = ('realized_lgd','mean'),
        avg_realized_loss = ('realized_loss','mean'),
        avg_ead = ('ead','mean')
    ).reset_index()

lgd_by_grade

,grade,defaulted_loan_count,avg_realized_lgd,avg_realized_loss,avg_ead
0,A,460,0.531056,7011.446304,13186.793478
1,B,1704,0.558952,7761.744126,13588.292254
2,C,2924,0.611491,9378.598331,14892.886457
3,D,1997,0.647012,10760.132439,16226.952929
4,E,1159,0.695648,12972.271803,18428.515962
5,F,439,0.716095,13736.755080,18962.015945
6,G,177,0.766419,15456.646328,20283.192090


## Assign LGD Estimates

Each borrower receives an LGD estimate based on their credit grade.

If a grade-level estimate is unavailable, the global LGD average is used as a fallback estimate.

In [65]:
risk_df=risk_df.merge(lgd_by_grade[['grade','avg_realized_lgd']], on='grade', how='left')

risk_df['lgd_estimate'] = risk_df['avg_realized_lgd'].fillna(global_lgd)

risk_df[['grade','realized_lgd', 'lgd_estimate']].tail()

,grade,realized_lgd,lgd_estimate
39995,A,0.243054,0.531056
39996,D,0.785408,0.647012
39997,A,0.000000,0.531056
39998,C,0.616613,0.611491
39999,D,0.727142,0.647012


## Calculate Expected Loss

Expected Loss combines the three major credit risk components:

Expected Loss = PD × LGD × EAD

where:

- PD = Probability of Default
- LGD = Loss Given Default
- EAD = Exposure at Default

Expected Loss estimates the average financial loss expected from each borrower.

In [66]:
risk_df['expected_loss'] = risk_df['pd_score'] * risk_df['lgd_estimate'] * risk_df['ead']

risk_df[['pd_score', 'lgd_estimate', 'ead', 'expected_loss']].head()

,pd_score,lgd_estimate,ead,expected_loss
0,0.495415,0.695648,12000,4135.613883
1,0.504460,0.611491,20100,6200.299219
2,0.301881,0.558952,8000,1349.895463
3,0.685799,0.611491,23325,9781.568456
4,0.739917,0.716095,21250,11259.330155


## Estimate Risk-Adjusted Return

Expected interest income is estimated using:

Interest Income = EAD × Interest Rate

Risk-adjusted return is then calculated by subtracting expected losses from expected interest income.

This provides a simplified estimate of portfolio profitability after accounting for credit risk.

In [67]:
risk_df['interest_rate_decimal'] = risk_df['int_rate']/100
risk_df['expected_interest_income'] = risk_df['ead'] * risk_df['interest_rate_decimal']
risk_df['risk_adjusted_return'] = risk_df['expected_interest_income'] - risk_df['expected_loss']

risk_df[['int_rate', 'ead', 'expected_interest_income', 'risk_adjustment_return','expected_loss', 'expected_interest_income']].head()

,int_rate,ead,expected_interest_income,risk_adjustment_return,expected_loss,expected_interest_income
0,19.89,12000,2386.8000,-7.711748e+07,4135.613883,2386.8000
1,15.59,20100,3133.5900,-9.509254e+07,6200.299219,3133.5900
2,11.14,8000,891.2000,-1.874409e+07,1349.895463,891.2000
3,12.69,23325,2959.9425,-1.500196e+08,9781.568456,2959.9425
4,22.95,21250,4876.8750,-2.159813e+08,11259.330155,4876.8750


## Expected Loss by Risk Band

Loss metrics are summarized across the previously created borrower risk bands.

The analysis includes:

- borrower count
- default rate
- average LGD
- average EAD
- expected loss
- risk-adjusted return

This validates whether higher-risk borrowers generate larger expected losses.

In [68]:
loss_by_risk_band = risk_df.groupby('risk_band')\
    .agg(
        borrower_count = ('actual_default_flag','count'),
        actual_default_rate = ('actual_default_flag','mean'),
        total_lgd = ('lgd_estimate','sum'),
        avg_lgd = ('lgd_estimate','mean'),
        total_ead = ('ead','sum'),
        avg_ead = ('ead','mean'),
        total_expected_loss = ('expected_loss','sum'),
        avg_expected_loss = ('expected_loss','mean'),
        total_risk_adjusted_return = ('risk_adjusted_return','sum'),
        avg_risk_adjusted_return = ('risk_adjusted_return','mean'),
    ).reset_index()

loss_by_risk_band

/var/folders/99/nqyz4s796bx09s6p71mvxd5h0000gn/T/ipykernel_3070/1092963251.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  loss_by_risk_band = risk_df.groupby('risk_band')\


,risk_band,borrower_count,actual_default_rate,total_lgd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_risk_adjusted_return,avg_risk_adjusted_return
0,Very Low,8000,0.048625,4334.087003,0.541761,108462950,13557.868750,9.032942e+06,1129.117775,-2.344783e+05,-29.309787
1,Low,8000,0.107500,4572.033718,0.571504,101339225,12667.403125,1.710190e+07,2137.737331,-5.719008e+06,-714.876028
2,Medium,8000,0.184125,4790.797142,0.598850,108561100,13570.137500,2.753516e+07,3441.894746,-1.297154e+07,-1621.442993
3,High,8000,0.263875,4998.268621,0.624784,122673975,15334.246875,4.335050e+07,5418.812571,-2.408943e+07,-3011.178331
4,Very High,8000,0.503375,5221.813230,0.652727,139505050,17438.131250,6.951615e+07,8689.518128,-4.340289e+07,-5425.361732


## Expected Loss by Loan Purpose

Expected loss is summarized across different loan purposes.

This analysis helps identify which loan purposes contribute the highest credit risk and financial losses.

In [69]:
loss_by_purpose = risk_df.groupby('purpose')\
    .agg(
        borrower_count = ('actual_default_flag','count'),
        actual_default_rate = ('actual_default_flag','mean'),
        total_lgd = ('lgd_estimate','sum'),
        avg_lgd = ('lgd_estimate','mean'),
        total_ead = ('ead','sum'),
        avg_ead = ('ead','mean'),
        total_expected_loss = ('expected_loss','sum'),
        avg_expected_loss = ('expected_loss','mean'),
        total_risk_adjusted_return = ('risk_adjusted_return','sum'),
        avg_risk_adjusted_return = ('risk_adjusted_return','mean'),
    ).reset_index()

loss_by_purpose

,purpose,borrower_count,actual_default_rate,total_lgd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_risk_adjusted_return,avg_risk_adjusted_return
0,car,441,0.192744,258.236215,0.585570,3978100,9020.634921,1.056742e+06,2396.239916,-5.407931e+05,-1226.288159
1,credit_card,8738,0.189517,5086.346075,0.582095,128972525,14759.959373,3.197287e+07,3659.060626,-1.619824e+07,-1853.769577
2,debt_consolidation,23248,0.232364,13996.406497,0.602048,358433775,15417.832717,1.071717e+08,4609.932045,-5.621371e+07,-2418.001969
3,educational,10,0.100000,5.726864,0.572686,64100,6410.000000,9.609548e+03,960.954823,-2.402728e+03,-240.272823
4,home_improvement,2564,0.194228,1520.442383,0.592996,36308925,14161.047192,9.620320e+06,3752.075007,-4.719078e+06,-1840.513855
5,house,212,0.231132,131.184917,0.618797,3284400,15492.452830,1.051383e+06,4959.351747,-5.433927e+05,-2563.173221
6,major_purchase,874,0.209382,516.627772,0.591107,10018450,11462.757437,2.899591e+06,3317.609565,-1.524328e+06,-1744.082892
7,medical,490,0.255102,297.071660,0.606269,4503725,9191.275510,1.365519e+06,2786.774273,-7.095847e+05,-1448.131993
8,moving,298,0.281879,183.428387,0.615531,2363725,7931.963087,7.479000e+05,2509.731646,-3.811504e+05,-1279.028349
9,other,2279,0.235630,1396.931225,0.612958,22474075,9861.375603,7.128134e+06,3127.746533,-3.673398e+06,-1611.846275


## Expected Loss by Credit Grade

Expected loss metrics are summarized across Lending Club credit grades.

The analysis evaluates:

- borrower count
- default rate
- average LGD
- average EAD
- expected loss
- risk-adjusted return

This provides insight into how credit quality affects portfolio risk.

In [70]:
loss_by_grade = risk_df.groupby('grade')\
    .agg(
        borrower_count = ('actual_default_flag','count'),
        actual_default_rate = ('actual_default_flag','mean'),
        total_lgd = ('lgd_estimate','sum'),
        avg_lgd = ('lgd_estimate','mean'),
        total_ead = ('ead','sum'),
        avg_ead = ('ead','mean'),
        total_expected_loss = ('expected_loss','sum'),
        avg_expected_loss = ('expected_loss','mean'),
        total_risk_adjusted_return = ('risk_adjusted_return','sum'),
        avg_risk_adjusted_return = ('risk_adjusted_return','mean'),
    ).reset_index()

loss_by_grade

,grade,borrower_count,actual_default_rate,total_lgd,avg_lgd,total_ead,avg_ead,total_expected_loss,avg_expected_loss,total_risk_adjusted_return,avg_risk_adjusted_return
0,A,6741,0.068239,3579.847018,0.531056,93471175,13866.069574,8.801781e+06,1305.708462,-2.135850e+06,-316.844668
1,B,11515,0.147981,6436.330728,0.558952,151690400,13173.287017,2.983988e+07,2591.392215,-1.364311e+07,-1184.811983
2,C,11544,0.253292,7059.048015,0.611491,165048100,14297.305960,5.199048e+07,4503.680062,-2.877665e+07,-2492.779585
3,D,6072,0.328887,3928.656358,0.647012,94568300,15574.489460,3.806217e+07,6268.472925,-2.121796e+07,-3494.393145
4,E,2877,0.402850,2001.380230,0.695648,51301550,17831.612791,2.467589e+07,8576.953065,-1.377870e+07,-4789.259335
5,F,935,0.469519,669.548931,0.716095,17717900,18949.625668,9.274417e+06,9919.162940,-4.848409e+06,-5185.463665
6,G,316,0.560127,242.188433,0.766419,6744875,21344.541139,3.892021e+06,12316.521398,-2.016681e+06,-6381.901049


## Save Outputs

The completed LGD, EAD, and Expected Loss outputs are saved for later analysis.

Saved files include:

- borrower-level Expected Loss dataset
- LGD summary by grade
- Expected Loss by risk band
- Expected Loss by grade
- Expected Loss by loan purpose

These outputs can be used for reporting, portfolio optimization, and regulatory risk analysis.

In [71]:
risk_df.to_pickle("../data/lgd_ead_expected_loss_output.pkl")

lgd_by_grade.to_csv("../data/lgd_by_grade.csv", index=False)
loss_by_risk_band.to_csv("../data/expected_loss_by_risk_band.csv", index=False)
loss_by_grade.to_csv("../data/expected_loss_by_grade.csv", index=False)
loss_by_purpose.to_csv("../data/expected_loss_by_purpose.csv", index=False)